# Representation Geometry: How the Model Organizes Concepts in Space

Large language models build internal representations of text as high-dimensional vectors at each layer. But what structure do these representations have? **Principal Component Analysis (PCA)** lets us project these high-dimensional hidden states into 2D space, revealing how the model organizes different semantic categories.

This notebook applies PCA to hidden states captured via **PyTorch hooks** at multiple layers of Llama 3.1 8B. We pass prompts from three distinct semantic categories (animals, science, geography) and observe:

- **Cluster formation**: Do tokens from the same category group together?
- **Layer progression**: How does the geometry change from early to late layers?
- **Variance structure**: How concentrated is the information in the top principal components?
- **Category differentiation**: How do cosine similarities between categories evolve across layers?

Based on general dimensionality reduction techniques applied to LLMs, and inspired by Reif et al., "Visualizing and Measuring the Geometry of BERT" (NeurIPS 2019).

## 1. Setup: Imports and Model Loading

In [ ]:
%matplotlib inline

import torch
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Configuration
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CACHE_DIR = os.path.expanduser("~/models")
OUTPUT_DIR = os.path.expanduser("~/hook_outputs/charts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model loaded successfully on: {}".format(model.device if hasattr(model, 'device') else 'multiple devices'))

## 2. Define Prompts by Semantic Category

We use three categories with three prompts each. This gives us enough data to see whether the model's internal representations reflect semantic groupings.

In [ ]:
# Define prompts organized by semantic category
categories = {
    "Animals": [
        "A cat sits on the mat",
        "The dog barked loudly",
        "Birds fly south in winter",
    ],
    "Science": [
        "Atoms are made of protons",
        "DNA contains genetic code",
        "Gravity pulls objects down",
    ],
    "Geography": [
        "Paris is in France",
        "Tokyo is the capital of Japan",
        "The Amazon is the longest river",
    ],
}

# Colors for each category
category_colors = {
    "Animals": "blue",
    "Science": "green",
    "Geography": "red",
}

# Layers to hook into (representative samples across the 32-layer model)
TARGET_LAYERS = [0, 8, 16, 24, 31]

print("Categories: {}".format(list(categories.keys())))
print("Prompts per category: {}".format(len(list(categories.values())[0])))
print("Target layers: {}".format(TARGET_LAYERS))

## 3. Capture Hidden States with PyTorch Hooks

We register forward hooks on the target layers to capture hidden states for every token across all prompts. Each hidden state vector has dimensionality 4096 (the model's hidden size).

In [ ]:
# Storage for hidden states: {layer_idx: list of (hidden_state_tensor, category_label)}
hidden_states_by_layer = {layer: [] for layer in TARGET_LAYERS}
category_labels_by_layer = {layer: [] for layer in TARGET_LAYERS}

# Create hook functions for each target layer
hooks = []

def make_hook_fn(layer_idx, category_name):
    """Create a hook function that captures hidden states for a specific layer."""
    def hook_fn(module, input, output):
        # output is a tuple; first element is the hidden state tensor
        hidden = output[0]  # shape: (batch_size, seq_len, hidden_dim) or (seq_len, hidden_dim)
        # Detach and move to CPU, store all token representations
        states = hidden.detach().cpu().float().numpy()
        if hidden.dim() == 3:
            for token_idx in range(states.shape[1]):
                hidden_states_by_layer[layer_idx].append(states[0, token_idx, :])
                category_labels_by_layer[layer_idx].append(category_name)
        elif hidden.dim() == 2:
            for token_idx in range(states.shape[0]):
                hidden_states_by_layer[layer_idx].append(states[token_idx, :])
                category_labels_by_layer[layer_idx].append(category_name)
    return hook_fn

# Process each category and prompt
for category_name, prompts in categories.items():
    for prompt in prompts:
        print("Processing [{}]: '{}'".format(category_name, prompt))

        # Register hooks for target layers
        hooks = []
        for layer_idx in TARGET_LAYERS:
            layer_module = model.model.layers[layer_idx]
            hook = layer_module.register_forward_hook(
                make_hook_fn(layer_idx, category_name)
            )
            hooks.append(hook)

        # Run the prompt through the model
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            _ = model(**inputs)

        # Remove hooks after each prompt
        for hook in hooks:
            hook.remove()

print("\nHidden states captured!")
for layer_idx in TARGET_LAYERS:
    print("  Layer {}: {} token representations".format(
        layer_idx, len(hidden_states_by_layer[layer_idx])
    ))

## 4. Apply PCA and Prepare Data

We fit PCA with 2 components for the scatter plots and 20 components for the variance analysis. This reduces each 4096-dimensional hidden state to a manageable number of dimensions for visualization.

In [ ]:
# Apply PCA at each layer
pca_2d_results = {}   # {layer: (projected_2d, labels)}
pca_20d_results = {}  # {layer: pca_model fitted with 20 components}

for layer_idx in TARGET_LAYERS:
    # Stack all hidden states for this layer into a matrix
    states_matrix = np.stack(hidden_states_by_layer[layer_idx])  # (n_tokens, hidden_dim)
    labels = category_labels_by_layer[layer_idx]

    # PCA with 2 components for scatter plots
    pca_2 = PCA(n_components=2)
    projected_2d = pca_2.fit_transform(states_matrix)
    pca_2d_results[layer_idx] = (projected_2d, labels, pca_2)

    # PCA with 20 components for variance analysis
    pca_20 = PCA(n_components=20)
    pca_20.fit(states_matrix)
    pca_20d_results[layer_idx] = pca_20

    print("Layer {} - Shape: {}, Explained variance (PC1+PC2): {:.1f}%".format(
        layer_idx,
        states_matrix.shape,
        pca_2.explained_variance_ratio_.sum() * 100
    ))

## 5. Visualization 1: PCA Scatter Plots Across Layers

Each subplot shows token representations projected into 2D PCA space at a different layer. Points are colored by semantic category. We expect to see clusters becoming more distinct in deeper layers as the model builds richer semantic representations.

In [ ]:
# Visualization 1: 2x3 grid of PCA scatter plots (one per layer + legend)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes_flat = axes.flatten()

for idx, layer_idx in enumerate(TARGET_LAYERS):
    ax = axes_flat[idx]
    projected_2d, labels, pca_model = pca_2d_results[layer_idx]

    # Plot each category separately for legend
    for cat_name, color in category_colors.items():
        mask = [l == cat_name for l in labels]
        mask_indices = [i for i, m in enumerate(mask) if m]
        cat_points = projected_2d[mask_indices]
        ax.scatter(
            cat_points[:, 0], cat_points[:, 1],
            c=color, label=cat_name, alpha=0.6, s=40, edgecolors="white", linewidth=0.5
        )

    var_explained = pca_model.explained_variance_ratio_.sum() * 100
    ax.set_title("Layer {} (var explained: {:.1f}%)".format(layer_idx, var_explained), fontsize=13, fontweight="bold")
    ax.set_xlabel("PC1 ({:.1f}%)".format(pca_model.explained_variance_ratio_[0] * 100))
    ax.set_ylabel("PC2 ({:.1f}%)".format(pca_model.explained_variance_ratio_[1] * 100))
    ax.legend(fontsize=9, loc="best")
    ax.grid(True, alpha=0.3)

# Use the last subplot for a summary legend
axes_flat[5].axis("off")
axes_flat[5].text(0.5, 0.6, "PCA Projection of\nHidden States", fontsize=16,
                  fontweight="bold", ha="center", va="center", transform=axes_flat[5].transAxes)
axes_flat[5].text(0.5, 0.35,
    "Each point = one token\nColored by semantic category\n\n"
    "Blue = Animals\nGreen = Science\nRed = Geography",
    fontsize=12, ha="center", va="center", transform=axes_flat[5].transAxes)

plt.suptitle("Representation Geometry Across Layers (Llama 3.1 8B)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "chart_23_pca_layers.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## 6. Visualization 2: Explained Variance by PCA Components

This plot shows how much variance is captured by each of the first 20 principal components at each layer. A steep drop-off means the representation is low-dimensional (most information lives in a few directions). A flatter curve suggests more distributed, higher-dimensional structure.

In [ ]:
# Visualization 2: Explained variance ratio for first 20 PCA components at each layer
fig, ax = plt.subplots(figsize=(12, 6))

layer_styles = {0: "o-", 8: "s-", 16: "^-", 24: "D-", 31: "v-"}
layer_cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(TARGET_LAYERS)))

for idx, layer_idx in enumerate(TARGET_LAYERS):
    pca_model = pca_20d_results[layer_idx]
    variance_ratios = pca_model.explained_variance_ratio_ * 100
    components = range(1, len(variance_ratios) + 1)
    ax.plot(
        components, variance_ratios,
        layer_styles[layer_idx],
        color=layer_cmap[idx],
        label="Layer {}".format(layer_idx),
        markersize=6, linewidth=2, alpha=0.8
    )

ax.set_xlabel("Principal Component", fontsize=13)
ax.set_ylabel("Explained Variance (%)", fontsize=13)
ax.set_title("Explained Variance by PCA Component Across Layers (Llama 3.1 8B)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="upper right")
ax.set_xticks(range(1, 21))
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 20.5)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_24_variance_explained.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## 7. Visualization 3: Inter-Category Cosine Similarity Across Layers

We compute the mean hidden state for each category at each layer, then measure the cosine similarity between each pair of categories. If the model differentiates categories, we expect cosine similarity to decrease in deeper layers as representations become more specialized.

In [ ]:
# Visualization 3: Cosine similarity between mean representations of category pairs
category_pairs = [
    ("Animals", "Science"),
    ("Animals", "Geography"),
    ("Science", "Geography"),
]

pair_similarities = {"{}-{}".format(a, b): [] for a, b in category_pairs}

for layer_idx in TARGET_LAYERS:
    # Compute mean representation for each category at this layer
    category_means = {}
    for cat_name in categories.keys():
        mask = [l == cat_name for l in category_labels_by_layer[layer_idx]]
        mask_indices = [i for i, m in enumerate(mask) if m]
        cat_states = np.stack([hidden_states_by_layer[layer_idx][i] for i in mask_indices])
        category_means[cat_name] = cat_states.mean(axis=0).reshape(1, -1)

    # Compute cosine similarity for each pair
    for cat_a, cat_b in category_pairs:
        sim = cosine_similarity(category_means[cat_a], category_means[cat_b])[0, 0]
        pair_similarities["{}-{}".format(cat_a, cat_b)].append(sim)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

pair_styles = {"Animals-Science": "o-", "Animals-Geography": "s-", "Science-Geography": "^-"}
pair_colors = {"Animals-Science": "purple", "Animals-Geography": "darkorange", "Science-Geography": "teal"}

for pair_name in pair_similarities:
    ax.plot(
        TARGET_LAYERS, pair_similarities[pair_name],
        pair_styles[pair_name],
        color=pair_colors[pair_name],
        label=pair_name,
        markersize=8, linewidth=2.5, alpha=0.85
    )

ax.set_xlabel("Layer", fontsize=13)
ax.set_ylabel("Cosine Similarity", fontsize=13)
ax.set_title("Inter-Category Cosine Similarity Across Layers (Llama 3.1 8B)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="best")
ax.set_xticks(TARGET_LAYERS)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_25_category_similarity.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## Summary

This notebook explored the **geometry of hidden state representations** in Llama 3.1 8B using PCA applied to hook-captured activations:

1. **PCA Scatter Plots (Chart 23)**: Projecting hidden states into 2D reveals how the model organizes tokens from different semantic categories. Early layers show mixed, overlapping representations, while deeper layers tend to show more distinct clustering by category -- the model progressively builds semantic structure.

2. **Explained Variance (Chart 24)**: The distribution of variance across principal components tells us about the intrinsic dimensionality of representations. If the first few components capture most variance, the representation is effectively low-dimensional. Different layers may have different variance profiles as the model reshapes its representational geometry.

3. **Inter-Category Cosine Similarity (Chart 25)**: By tracking how similar the mean representations of each category pair are across layers, we can see the model actively differentiating semantic categories. Decreasing cosine similarity in deeper layers indicates the model is pushing different concepts apart in representation space.

**Key insight**: The transformer does not just passively pass information through layers -- it actively restructures the geometry of its representations, progressively organizing them in ways that reflect semantic meaning. This geometric perspective, inspired by Reif et al. (2019), provides a complementary view to the logit lens and attention-based analyses.